# Rich Python API Construction

This notebook mirrors the richer engineering detail of notebook 2, but builds the full BattINFO chain directly in Python.

It demonstrates:

- constructing a rich `CellSpecification` in code
- inspecting electrodes, coating components, electrolyte, and separator
- deriving a lighter public `CellType`
- creating `CellInstance`, `Test`, and `Dataset` objects
- publishing and reloading `battinfo.publish.jsonld`


In [ ]:
import tempfile
from pathlib import Path
from pprint import pprint

from battinfo import (
    Coating,
    CellInstance,
    CellSpecification,
    CellType,
    CurrentCollector,
    Dataset,
    Electrode,
    Electrolyte,
    MaterialComponent,
    Salt,
    Separator,
    SolventMixture,
    Test,
    load_publication,
    publish,
)
from battinfo.bundle import ProvenanceInfo

workspace = Path(tempfile.mkdtemp(prefix="battinfo-rich-python-api-"))
dataset_dir = workspace / "a123-anr26650m1-b-demo"
raw_dir = dataset_dir / "timeseries" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
(raw_dir / "run-a.csv").write_text("time,voltage\n0,3.3\n1,3.2\n", encoding="utf-8")

print("Workspace:", workspace)
print("Dataset dir:", dataset_dir)


## Build A Rich CellSpecification

In [ ]:
spec = CellSpecification(
    id="https://w3id.org/battinfo/cell-type/9qfb-4wrn-ynwc-ayjw",
    manufacturer="A123",
    model="ANR26650M1-B",
    format="cylindrical",
    chemistry="Li-ion",
    positive_electrode_basis="LFP",
    negative_electrode_basis="graphite",
    size_code="R26650",
    properties={
        "nominal_capacity": {"typical_value": 2.5, "unit": "Ah"},
        "nominal_voltage": {"value": 3.3, "unit": "V"},
        "continuous_discharging_current": {"max_value": 50.0, "unit": "A"},
        "diameter": {"value": 26.0, "unit": "mm"},
        "height": {"value": 65.2, "unit": "mm"},
        "mass": {"value": 76.0, "unit": "g"},
    },
    positive_electrode=Electrode(
        coating=Coating(
            component={
                "active_material": [
                    MaterialComponent(
                        name="LFP",
                        property={
                            "mass_fraction": {"value": 0.94, "unit": "1"},
                            "particle_d50": {"value": 1.8, "unit": "um"},
                        },
                    )
                ],
                "binder": [
                    MaterialComponent(
                        name="PVDF",
                        property={"mass_fraction": {"value": 0.03, "unit": "1"}},
                    )
                ],
                "additive": [
                    MaterialComponent(
                        name="Carbon black",
                        property={"mass_fraction": {"value": 0.03, "unit": "1"}},
                    ),
                    MaterialComponent(
                        name="Al2O3 coating particles",
                        property={"mass_fraction": {"value": 0.005, "unit": "1"}},
                    ),
                ],
            },
            property={
                "loading": {"value": 20.0, "unit": "mg/cm^2"},
                "porosity": {"value": 0.35, "unit": "1"},
                "thickness": {"value": 66.0, "unit": "um"},
            },
            comment="Representative production coating composition.",
        ),
        current_collector=CurrentCollector(
            name="Al foil",
            property={
                "thickness": {"value": 15.0, "unit": "um"},
                "areal_mass": {"value": 40.5, "unit": "mg/cm^2"},
            },
            comment="Etched aluminum collector.",
        ),
        property={"compaction_density": {"value": 2.9, "unit": "g/cm^3"}},
        comment="Positive electrode parameters from internal recipe.",
    ),
    negative_electrode=Electrode(
        coating=Coating(
            component={
                "active_material": [
                    MaterialComponent(
                        name="Graphite",
                        property={"mass_fraction": {"value": 0.95, "unit": "1"}},
                    )
                ],
                "binder": [
                    MaterialComponent(
                        name="CMC/SBR",
                        property={"mass_fraction": {"value": 0.04, "unit": "1"}},
                    )
                ],
                "additive": [
                    MaterialComponent(
                        name="Carbon black",
                        property={"mass_fraction": {"value": 0.01, "unit": "1"}},
                    )
                ],
            },
            property={
                "loading": {"value": 11.2, "unit": "mg/cm^2"},
                "thickness": {"value": 78.0, "unit": "um"},
            },
            comment="Representative negative coating.",
        ),
        current_collector=CurrentCollector(
            name="Cu foil",
            property={"thickness": {"value": 10.0, "unit": "um"}},
        ),
        comment="Negative electrode with graphite-dominant active material.",
    ),
    electrolyte=Electrolyte(
        family="organic",
        solvent_mixture=SolventMixture(
            component=[
                MaterialComponent(name="EC", property={"volume_fraction": {"value": 0.3, "unit": "1"}}),
                MaterialComponent(name="EMC", property={"volume_fraction": {"value": 0.7, "unit": "1"}}),
            ],
            property={"water_content": {"max_value": 20, "unit": "ppm"}},
            comment="Base solvent system.",
        ),
        salt=Salt(
            name="LiPF6",
            cation="Li+",
            anion="PF6-",
            property={"concentration": {"value": 1.0, "unit": "mol/L"}},
            comment="Standard 1M salt concentration.",
        ),
        additive=[
            MaterialComponent(name="VC", property={"volume_fraction": {"value": 0.02, "unit": "1"}}),
            MaterialComponent(name="FEC", property={"volume_fraction": {"value": 0.03, "unit": "1"}}),
        ],
        property={
            "conductivity": {"value": 10.5, "unit": "mS/cm"},
            "viscosity": {"value": 2.4, "unit": "mPa*s"},
        },
        comment="Electrolyte composition at 25 C.",
    ),
    separator=Separator(
        material="PE/PP trilayer",
        property={
            "thickness": {"value": 20.0, "unit": "um"},
            "porosity": {"value": 0.42, "unit": "1"},
        },
        comment="Microporous polyolefin separator.",
    ),
    specification_comment=[
        "Primary review artifact for BattINFO/BDF battery descriptor structure.",
        "Values are representative and intended for schema/mapping review.",
    ],
    source=ProvenanceInfo(
        type="measurement",
        name="Catenaro 2021 ingestion pilot",
        file="ddata/a123/anr26650m1-b/catenaro-2021/battery.json",
        url="https://doi.org/10.17632/kxsbr4x3j2.2",
        retrieved_at=1772556000,
        workflow_version="battinfo-ingest-0.2.0",
        comment="Manually curated example promoted into the reusable BattINFO cell-type library.",
    ),
    comment=[
        "Reusable BattINFO library example for a curated commercial cylindrical cell type.",
        "Promoted from the canonical A123 review descriptor.",
    ],
)

pprint(spec.model_dump(exclude_none=True))


## Top-Level Summary

In [ ]:
summary = {
    "id": spec.id,
    "manufacturer": spec.manufacturer,
    "model": spec.model,
    "format": spec.format,
    "chemistry": spec.chemistry,
    "positive_electrode_basis": spec.positive_electrode_basis,
    "negative_electrode_basis": spec.negative_electrode_basis,
    "size_code": spec.size_code,
    "property_count": len(spec.properties),
    "example_property_keys": sorted(spec.properties)[:10],
}

pprint(summary)


In [ ]:
key_properties = {
    key: spec.properties[key]
    for key in [
        "nominal_capacity",
        "nominal_voltage",
        "continuous_discharging_current",
        "diameter",
        "height",
        "mass",
    ]
    if key in spec.properties
}

pprint(key_properties)


## Electrode Helpers

In [ ]:
def flatten_components(component_block: dict[str, list[MaterialComponent]]) -> list[dict]:
    rows = []
    for role, items in component_block.items():
        for item in items:
            rows.append(
                {
                    "role": role,
                    "name": item.name,
                    "property": item.property,
                    "comment": item.comment,
                }
            )
    return rows

def summarize_electrode(name: str, electrode: Electrode | None) -> dict:
    if electrode is None:
        return {"name": name, "comment": None, "coating_property": {}, "current_collector": None, "components": []}
    return {
        "name": name,
        "comment": electrode.comment,
        "coating_property": electrode.coating.property if electrode.coating is not None else {},
        "current_collector": electrode.current_collector.model_dump(exclude_none=True) if electrode.current_collector is not None else None,
        "components": flatten_components(electrode.coating.component) if electrode.coating is not None else [],
    }


## Positive Electrode

In [ ]:
positive_summary = summarize_electrode("positive", spec.positive_electrode)
pprint(positive_summary)


## Negative Electrode

In [ ]:
negative_summary = summarize_electrode("negative", spec.negative_electrode)
pprint(negative_summary)


## Electrolyte And Separator

In [ ]:
electrolyte = spec.electrolyte
separator = spec.separator

solvent_mixture = electrolyte.solvent_mixture if electrolyte is not None else None
electrolyte_summary = {
    "family": electrolyte.family if electrolyte is not None else None,
    "solvent_components": [item.model_dump(exclude_none=True) for item in solvent_mixture.component] if solvent_mixture is not None else [],
    "solvent_properties": solvent_mixture.property if solvent_mixture is not None else {},
    "salt": electrolyte.salt.model_dump(exclude_none=True) if electrolyte is not None and electrolyte.salt is not None else None,
    "additives": [item.model_dump(exclude_none=True) for item in electrolyte.additive] if electrolyte is not None else [],
    "bulk_properties": electrolyte.property if electrolyte is not None else {},
    "comment": electrolyte.comment if electrolyte is not None else None,
}

separator_summary = {
    "material": separator.material if separator is not None else None,
    "property": separator.property if separator is not None else {},
    "comment": separator.comment if separator is not None else None,
}

pprint({"electrolyte": electrolyte_summary, "separator": separator_summary})


## Derive The Lightweight Public CellType

In [ ]:
cell_type = CellType.from_cell_specification(spec)

pprint(cell_type.model_dump(exclude_none=True))


## Build The Publication Chain

In [ ]:
cell = CellInstance(
    cell_type=cell_type,
    serial_number="A123-ANR26650M1-B-DEMO-001",
)

test = Test(
    cell=cell,
    kind="cycle_life",
    protocol="0.5C baseline cycling",
    instrument="Arbin cycler",
    status="completed",
)

dataset = Dataset(
    path=str(dataset_dir),
    cell=cell,
    test=test,
    name="A123 ANR26650M1-B cycling dataset",
    description="Synthetic demo dataset for a rich Python API notebook.",
)

pprint({
    "cell_type": cell_type.model_dump(exclude_none=True),
    "cell_instance": cell.model_dump(exclude_none=True),
    "test": test.model_dump(exclude_none=True),
    "dataset": dataset.model_dump(exclude_none=True),
})


## Publish And Reload

In [ ]:
report = publish(
    cell_specification=spec,
    cell_type=cell_type,
    cell_instance=cell,
    test=test,
    dataset=dataset,
)

bundle = load_publication(dataset_dir / "battinfo.publish.jsonld")

pprint(report)
pprint({
    "cell_specification_id": bundle.cell_specification.id if bundle.cell_specification else None,
    "cell_type_id": bundle.cell_type.id,
    "cell_instance_id": bundle.cell_instance.id,
    "test_protocol": bundle.test.protocol.name,
    "test_instrument": bundle.test.instrument,
    "dataset_access_url": bundle.dataset.access_url,
})
